In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/PredictionIpynb/Basic_train_test_split.ipynb"

In [ ]:
ModelName = "SVM"

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_classification(random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    random_state=0)
clf = SVC(random_state=0)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)
cm = confusion_matrix(y_test, predictions, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=clf.classes_)
disp.plot()

plt.show()


In [ ]:
class SVMSentiment(Base):
    """Predict fine-grained sentiment scores using a sklearn 
    linear Support Vector Machine (SVM) pipeline."""
    def __init__(self, model_file: str=None) -> None:
        super().__init__()
        from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
        from sklearn.linear_model import SGDClassifier
        from sklearn.pipeline import Pipeline
        self.pipeline = Pipeline(
            [
                ('vect', CountVectorizer()),
                ('tfidf', TfidfTransformer()),
                ('clf', SGDClassifier(
                    loss='hinge',
                    penalty='l2',
                    alpha=1e-3,
                    random_state=42,
                    max_iter=100,
                    learning_rate='optimal',
                    tol=None,
                )),
            ]
        )

    def predict(self, train_file: str, test_file: str, lower_case: bool=False) -> pd.DataFrame:
        "Train model using sklearn pipeline"
        train_df = self.read_data(train_file, lower_case)
        learner = self.pipeline.fit(train_df['text'], train_df['truth'])
        # Predict class labels using the learner and output DataFrame
        test_df = self.read_data(test_file, lower_case)
        test_df['pred'] = learner.predict(test_df['text'])
        return test_df

In [ ]:
hyper_params = { 
    "lr": 0.35,         # Learning rate
    "epoch": 100,       # Number of training epochs to train for
    "wordNgrams": 3,    # Number of word n-grams to consider during training
    "dim": 155,         # Size of word vectors
    "ws": 5,            # Size of the context window for CBOW or skip-gram
    "minn": 2,          # Min length of char ngram
    "maxn": 5,          # Max length of char ngram
    "bucket": 2014846,  # Number of buckets
}

def trainer(filepath: str, train_file: str, model_path: str, hyper_params: dict) -> None:
    "Train sentiment model using FastText: https://fasttext.cc/docs/en/supervised-tutorial.html"
    train = os.path.join(filepath, train_file)
    model = fasttext.train_supervised(input=train, **hyper_params)
    print("FastText model trained with hyperparameters: \n {}".format(hyper_params))
    # Save models to model directory for fasttext
    make_dirs(model_path)
    model.save_model(os.path.join(model_path, "sst5.bin"))
    # Quantize model to reduce space usage
    model.quantize(input=train, qnorm=True, retrain=True, cutoff=110539)
    model.save_model(os.path.join(model_path, "sst5.ftz"))

# Online Example

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv("../input/train.csv")
test = pd.read_csv("../input/test.csv")
sample_submission = pd.read_csv("../input/sampleSubmission.csv")
training_labels = LabelEncoder().fit_transform(train['target'])

# SVMs tend to like features that look similar to ~ N(0,1), so let's stabilise
# the long tails
train_features = train.drop('target', axis=1)
train_features[train_features > 4] = 4

model = LinearSVC().fit(train_features, training_labels)

scores = model.decision_function(test)
predictions = 1.0 / (1.0 + np.exp(-scores))
row_sums = predictions.sum(axis=1)
predictions_normalised = predictions / row_sums[:, np.newaxis]

# create submission file
prediction_DF = pd.DataFrame(predictions_normalised, index=sample_submission.id.values, columns=sample_submission.columns[1:])
prediction_DF.to_csv('svc_submission.csv', index_label='id')